# Python Foundations for AI & Machine Learning
---
# DAY 4: From Web Page to Insight

**The story.** You work as a data analyst for a used-car dealership in Pakistan. The owner asks:
*Which cars hold their value? Does the city change the price? What is a fair price for a 2018 sedan with 60,000 km?*

To answer, you will **collect** listings from the web (Module 27), **clean** the messy text into numbers (Module 28), and **explore** the market with statistics and charts (Module 29). First you need the two libraries that make this work: NumPy and Pandas.

---
# MODULE 25: NumPy

**NumPy** stores numbers in compact arrays and applies operations to **all elements at once** (*vectorisation*). It is the foundation under Pandas, scikit-learn and deep learning frameworks.

## 25.1 Arrays instead of lists

In [ ]:
import numpy as np
import time

prices_lacs = [28.5, 55.0, 42.5, 89.0, 36.0]       # a Python list (1 lac = 100,000 rupees)
prices = np.array(prices_lacs)                      # a NumPy array

print(prices)
print("dtype:", prices.dtype, "| shape:", prices.shape, "| dimensions:", prices.ndim, "| size:", prices.size)

# Ways to create arrays
print(np.arange(2015, 2026, 2))                     # like range(), but an array
print(np.linspace(0, 1, 5))                         # 5 evenly spaced numbers from 0 to 1
print(np.zeros(3), np.ones(3), np.full(3, 7.5))
rng = np.random.default_rng(42)                     # a seeded generator makes results repeatable
print(rng.integers(2005, 2027, size=5))

[28.5 55.  42.5 89.  36. ]
dtype: float64 | shape: (5,) | dimensions: 1 | size: 5
[2015 2017 2019 2021 2023 2025]
[0.   0.25 0.5  0.75 1.  ]
[0. 0. 0.] [1. 1. 1.] [7.5 7.5 7.5]
[2006 2022 2019 2014 2014]


In [ ]:
# Why bother? Speed. Add 10% to one million prices.
listing_prices = np.random.default_rng(1).uniform(5, 150, 1_000_000)
print(listing_prices)
as_list = listing_prices.tolist()

t0 = time.perf_counter()
with_loop = [p * 1.10 for p in as_list]             # Python loop
loop_time = time.perf_counter() - t0

t0 = time.perf_counter()
vectorised = listing_prices * 1.10                  # one NumPy operation
numpy_time = time.perf_counter() - t0

print(f"loop: {loop_time:.4f}s | numpy: {numpy_time:.4f}s | NumPy is ~{loop_time / numpy_time:,.0f}x faster")

[ 79.21413558 142.81723597  25.90314384 ... 105.19053854  91.33139151
 109.17248315]
loop: 0.1621s | numpy: 0.0069s | NumPy is ~24x faster


## 25.2 Vectorised arithmetic and `np.where`
Operations apply element by element. No loops needed.

In [ ]:
prices_pkr = prices * 100_000                       # lacs -> rupees for every car at once
transfer_fee = prices_pkr * 0.03                    # illustrative 3% transfer and registration cost
print((prices_pkr + transfer_fee).astype(int))

segment = np.where(prices < 30, "budget", np.where(prices < 60, "mid", "premium"))
print(segment)

# Car loan EMI for several tenures at once (illustrative 18% yearly interest)
principal, annual_rate = 2_000_000, 0.18
years = np.array([1, 2, 3, 4, 5])
r, n = annual_rate / 12, years * 12
emi = principal * r * (1 + r) ** n / ((1 + r) ** n - 1)
for y, e, total in zip(years, emi, emi * n):
    print(f"{y} year(s): EMI Rs. {e:,.0f} | total paid Rs. {total:,.0f} | interest Rs. {total - principal:,.0f}")

[2935500 5665000 4377500 9167000 3708000]
['budget' 'mid' 'mid' 'premium' 'mid']
1 year(s): EMI Rs. 183,360 | total paid Rs. 2,200,320 | interest Rs. 200,320
2 year(s): EMI Rs. 99,848 | total paid Rs. 2,396,357 | interest Rs. 396,357
3 year(s): EMI Rs. 72,305 | total paid Rs. 2,602,972 | interest Rs. 602,972
4 year(s): EMI Rs. 58,750 | total paid Rs. 2,820,000 | interest Rs. 820,000
5 year(s): EMI Rs. 50,787 | total paid Rs. 3,047,211 | interest Rs. 1,047,211


## 25.3 Indexing, slicing and boolean masks
A **mask** is an array of `True`/`False`. Using it as an index keeps only the `True` positions. This is how you filter data in NumPy **and** in Pandas.

In [ ]:
year = np.array([2012, 2018, 2020, 2016, 2022, 2019, 2015])
price = np.array([28, 55, 72, 41, 98, 63, 36.5])               # lacs
km = np.array([95_000, 62_000, 31_000, 88_000, 12_000, 45_000, 120_000])

print(year[0], year[-1], year[1:4], year[::2])                 # indexing and slicing, as with lists

recent = year >= 2018                                          # boolean mask
print(recent)
print("Prices of 2018+ cars:", price[recent])
print("2016+ and under 90,000 km:", price[(year >= 2016) & (km < 90_000)])     # combine with & | ~ (not and/or)

order = np.argsort(-price)                                     # positions that sort prices from high to low
print("Years of the 3 most expensive cars:", year[order][:3])

fuels, counts = np.unique(["Petrol", "Hybrid", "Petrol", "Diesel", "Petrol"], return_counts=True)
print(dict(zip(fuels.tolist(), counts.tolist())))

2012 2015 [2018 2020 2016] [2012 2020 2022 2015]
[False  True  True False  True  True False]
Prices of 2018+ cars: [55. 72. 98. 63.]
2016+ and under 90,000 km: [55. 72. 41. 98. 63.]
Years of the 3 most expensive cars: [2022 2020 2019]
{'Diesel': 1, 'Hybrid': 1, 'Petrol': 3}


## 25.4 Statistics along axes
`axis=0` works down the columns, `axis=1` across the rows.

In [ ]:
cars = np.column_stack([year, price, km])           # 7 cars x 3 columns
print(cars.shape)
print("column means (year, price, km):", cars.mean(axis=0).round(1))

print("median:", np.median(price), "| std:", price.std().round(2))
q1, q3 = np.percentile(price, [25, 75])
iqr = q3 - q1
outliers = (price < q1 - 1.5 * iqr) | (price > q3 + 1.5 * iqr)
print("IQR outlier mask:", outliers)

z_scores = (price - price.mean()) / price.std()      # standardisation: a core ML preprocessing step
print(z_scores.round(2))
print("correlation of year and price:", np.corrcoef(year, price).round(2))

(7, 3)
column means (year, price, km): [2.01740e+03 5.62000e+01 6.47143e+04]
median: 55.0 | std: 22.23
IQR outlier mask: [False False False False False False False]
[-1.27 -0.05  0.71 -0.68  1.88  0.31 -0.89]
correlation of year and price: [[1.   0.96]
 [0.96 1.  ]]


In [ ]:
# @title Default title text
from IPython.display import HTML, display

display(HTML(r'''
<style>
  .cx{background:#0f172a;color:#e2e8f0;font:13px monospace;padding:14px;border-radius:12px;max-width:1000px}
  .cx button{background:#1e293b;color:#e2e8f0;border:1px solid #475569;border-radius:8px;padding:4px 11px;margin:3px;font:13px monospace;cursor:pointer}
  .cx button:hover{border-color:#facc15}
  .cx .ctl{text-align:center;margin-bottom:6px}
  .cx .pills{display:flex;flex-wrap:wrap;gap:5px;justify-content:center;margin:4px 0 10px}
  .cx .pill{border:2px solid #334155;background:#1e293b;border-radius:14px;padding:2px 9px;font-size:11px;cursor:pointer;color:#64748b}
  .cx .pill.done{border-color:#166534;color:#4ade80}
  .cx .pill.on{border-color:#facc15;background:#facc15;color:#0f172a;font-weight:bold}
  .cx .main{display:flex;gap:12px;flex-wrap:wrap}
  .cx .left{flex:1 1 360px;min-width:290px}
  .cx .right{flex:1.2 1 420px;min-width:300px}
  .cx .ttl{font-size:11px;font-weight:bold;color:#94a3b8;margin:0 0 4px}
  .cx .code{background:#1e293b;border-radius:8px;padding:6px 0;font-size:12px;overflow:auto;max-height:420px}
  .cx .sl{padding:1px 8px;color:#e2e8f0;border-left:4px solid transparent;white-space:pre;min-height:1.45em}
  .cx .sl i{color:#475569;font-style:normal;display:inline-block;width:24px}
  .cx .sl .cm{color:#64748b}
  .cx .sl.old{color:#94a3b8}
  .cx .sl.on{color:#facc15;background:#334155;border-left-color:#facc15;animation:cxpop .5s ease-out}
  @keyframes cxpop{from{opacity:0;transform:translateX(-18px)}to{opacity:1;transform:none}}
  .cx .term{background:#020617;border-radius:8px;padding:8px 12px;margin-top:8px;min-height:60px;font-size:12px;line-height:1.55;overflow-x:auto}
  .cx .term .h{color:#64748b;font-size:11px;margin-bottom:2px}
  .cx .term .l{white-space:pre;color:#94a3b8}
  .cx .term .l.new{color:#4ade80;animation:cxpop .5s ease-out}
  .cx .stage{border:2px solid #334155;background:#111c33;border-radius:12px;padding:10px;min-height:330px;text-align:center}
  .cx .stt{font-size:12px;color:#facc15;font-weight:bold;margin-bottom:8px;text-align:left}
  .cx .fi{animation:cxfi .6s both}
  @keyframes cxfi{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:none}}
  .cx table.tb{border-collapse:separate;border-spacing:3px;margin:6px auto;font-size:12px}
  .cx .tb th{color:#94a3b8;font-weight:normal;font-size:11px;padding:2px 8px}
  .cx .tb td{border:2px solid #334155;background:#1e293b;border-radius:6px;padding:3px 9px;text-align:right}
  .cx .tb tr.hl td{border-color:#facc15;background:#334155;color:#facc15}
  .cx .tb tr.red td{border-color:#f87171;background:#2a1215;color:#f87171}
  .cx .tb tr.grn td{border-color:#166534}
  .cx .tb td.k{color:#38bdf8;text-align:left;font-weight:bold}
  .cx .chips{display:flex;flex-wrap:wrap;gap:5px;justify-content:center;margin:5px 0;align-items:center}
  .cx .cl{color:#94a3b8;font-size:11px;min-width:46px;text-align:right}
  .cx .chip{border:2px solid #475569;background:#1e293b;border-radius:8px;padding:2px 8px;font-size:12px;position:relative}
  .cx .chip small{display:block;color:#64748b;font-size:9px;text-align:center}
  .cx .chip.y{border-color:#facc15;color:#facc15}
  .cx .chip.r{border-color:#f87171;color:#f87171}
  .cx .chip.b{border-color:#38bdf8;color:#38bdf8}
  .cx .chip.g{border-color:#4ade80;color:#4ade80}
  .cx .eq{background:#1e293b;border:2px solid #475569;border-radius:8px;padding:5px 10px;margin:6px auto;display:inline-block;text-align:left;font-size:12px;line-height:1.7}
  .cx .eq b{color:#4ade80}
  .cx .eq .y{color:#facc15}
  .cx .big{font-size:20px;font-weight:bold;color:#4ade80;margin:6px 0}
  .cx .row2{display:flex;gap:10px;flex-wrap:wrap;justify-content:center;align-items:flex-start}
  .cx svg{width:100%;height:auto;display:block}
  .cx .note{background:#1e293b;border-left:4px solid #facc15;border-radius:6px;padding:8px 12px;margin-top:10px;font-size:13px;line-height:1.6;color:#cbd5e1;text-align:left;min-height:96px}
  .cx .note b{color:#fde68a}
  .cx .note code{color:#38bdf8}
</style>

<div class="cx">
  <div class="ctl">
    <button id="Cxb">◀ Back</button><button id="Cxn">Next ▶</button><button id="Cxp">⏵ Play</button><button id="Cxr">↺ Reset</button>
    <span id="Cxc" style="color:#94a3b8;font-size:13px"></span>
  </div>
  <div class="pills" id="Cxpills"></div>
  <div class="main">
    <div class="left">
      <div class="ttl">YOUR CODE (grows one line per step)</div>
      <div class="code" id="Cxcode"></div>
      <div class="term"><div class="h">OUTPUT SO FAR</div><div id="Cxterm"></div></div>
    </div>
    <div class="right">
      <div class="ttl">WHAT THIS LINE DOES</div>
      <div class="stage" id="Cxstage"></div>
    </div>
  </div>
  <div class="note" id="Cxnote"></div>
</div>

<script>
(function(){
  const $=id=>document.getElementById(id);
  const esc=s=>String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;');

  // ---------------- data (made up) ----------------
  const year=[2018,2019,2017,2020,2021,2016,2022];
  const price=[24,27,22,31,36,18,95];
  const km=[80000,70000,95000,52000,30000,120000,12000];
  const N=7;

  // ---------------- numpy-like maths ----------------
  const mean=a=>a.reduce((s,v)=>s+v,0)/a.length;
  const std=a=>{const m=mean(a);return Math.sqrt(mean(a.map(v=>(v-m)*(v-m))));};
  const sortd=a=>a.slice().sort((x,y)=>x-y);
  const pct=(a,p)=>{const s=sortd(a),pos=(s.length-1)*p/100,lo=Math.floor(pos),hi=Math.ceil(pos);return s[lo]+(pos-lo)*(s[hi]-s[lo]);};
  const median=a=>pct(a,50);
  const corr=(a,b)=>{const ma=mean(a),mb=mean(b);let c=0;for(let i=0;i<a.length;i++)c+=(a[i]-ma)*(b[i]-mb);return c/(a.length*std(a)*std(b));};
  const rnd=(x,d)=>{const m=Math.pow(10,d);return Math.round(x*m)/m;};
  const pyf=x=>Number.isInteger(x)?x.toFixed(1):String(x);
  function npArr(a){
    let p=0;
    a.forEach(v=>{for(let d=0;d<=8;d++){if(Math.abs(Number(v.toFixed(d))-v)<1e-9){p=Math.max(p,d);break;}}});
    const ss=a.map(v=>{let s=v.toFixed(p);if(p===0)return s+'.';s=s.replace(/0+$/,'');const f=s.length-s.indexOf('.')-1;return s+' '.repeat(p-f);});
    const w=Math.max.apply(null,ss.map(s=>s.length));
    return '['+ss.map(s=>s.padStart(w,' ')).join(' ')+']';
  }
  const boolArr=a=>'['+a.map(b=>b?' True':'False').join(' ')+']';

  // ---------------- computed values ----------------
  const cmean=[mean(year),mean(price),mean(km)];
  const mP=mean(price), sd=std(price), med=median(price), sp=sortd(price);
  const q1=pct(price,25), q3=pct(price,75), iqr=q3-q1, lowF=q1-1.5*iqr, upF=q3+1.5*iqr;
  const mask=price.map(p=>p<lowF||p>upF);
  const z=price.map(p=>(p-mP)/sd);
  const rr=corr(year,price);

  // ---------------- drawing helpers ----------------
  const fi=(i)=>' class="fi" style="animation-delay:'+(i*0.08)+'s"';
  function tbl(head,rows,rc){
    let h='<table class="tb"><tr>'+head.map(x=>'<th>'+x+'</th>').join('')+'</tr>';
    rows.forEach((r,i)=>{h+='<tr class="fi '+(rc?rc(i):'')+'" style="animation-delay:'+(i*0.08)+'s">'+r.map((c,j)=>'<td'+(j===0?' class="k"':'')+'>'+c+'</td>').join('')+'</tr>';});
    return h+'</table>';
  }
  const carRows=()=>year.map((y,i)=>['car '+i,y,price[i],km[i]]);
  const chips=(label,arr,cls,sub)=>'<div class="chips"><span class="cl">'+label+'</span>'+arr.map((v,i)=>'<span class="chip '+(cls?cls(i,v):'')+'">'+v+(sub?'<small>'+sub(i)+'</small>':'')+'</span>').join('')+'</div>';
  const eq=h=>'<div class="eq fi">'+h+'</div>';

  function nl(o){
    const L=30,R=610,y=84,sx=v=>L+(v/100)*(R-L);
    let s='<svg viewBox="0 0 640 160">';
    if(o.band) s+='<rect x="'+sx(o.band.a)+'" y="'+(y-13)+'" width="'+(sx(o.band.b)-sx(o.band.a))+'" height="26" rx="6" fill="'+o.band.c+'" opacity="0.28"/>';
    s+='<line x1="'+L+'" y1="'+y+'" x2="'+R+'" y2="'+y+'" stroke="#475569" stroke-width="2"/>';
    for(let t=0;t<=100;t+=10) s+='<line x1="'+sx(t)+'" y1="'+(y-4)+'" x2="'+sx(t)+'" y2="'+(y+4)+'" stroke="#64748b"/><text x="'+sx(t)+'" y="152" fill="#64748b" font-size="10" text-anchor="middle">'+t+'</text>';
    (o.marks||[]).forEach(m=>{
      const ty=14+(m.row||0)*14;
      s+='<line x1="'+sx(m.x)+'" y1="'+(ty+4)+'" x2="'+sx(m.x)+'" y2="'+y+'" stroke="'+m.c+'" stroke-width="2" stroke-dasharray="4 3"/>'+
         '<text x="'+sx(m.x)+'" y="'+ty+'" fill="'+m.c+'" font-size="11" font-weight="bold" text-anchor="middle">'+m.t+'</text>';
    });
    const ds=o.dots.map((v,i)=>({v:v,i:i})).sort((a,b)=>a.v-b.v);
    ds.forEach((d,k)=>{
      const c=o.col?o.col(d.i):'#38bdf8';
      s+='<circle cx="'+sx(d.v)+'" cy="'+y+'" r="7" fill="'+c+'" stroke="#0f172a" stroke-width="2"/>'+
         '<text x="'+sx(d.v)+'" y="'+(y+25+(k%2)*12)+'" fill="#94a3b8" font-size="10" text-anchor="middle">'+d.v+'</text>';
    });
    return s+'</svg>';
  }

  // ---------------- steps ----------------
  const STEPS=[
    {pill:'data',lines:['import numpy as np','','year  = np.array([2018, 2019, 2017, 2020, 2021, 2016, 2022])','price = np.array([24, 27, 22, 31, 36, 18, 95])          # lakh PKR','km    = np.array([80000, 70000, 95000, 52000, 30000, 120000, 12000])'],
     out:null,
     vis:()=>'<div class="stt">Setup (made up, your snippet did not show it)</div>'+chips('year',year,()=> 'b')+chips('price',price,(i)=>i===6?'r':'g')+chips('km',km,()=>'y')+'<div class="eq fi">three separate 1-D arrays, 7 cars each<br>position 0 of every array belongs to <b>car 0</b>, and so on</div>',
     note:'Not part of your snippet: three <b>1-D arrays</b> with one value per car. The same position in each array describes the same car. The last price (<b>95</b>) is deliberately an outlier so the later steps have something to find.'},

    {pill:'column_stack',lines:['cars = np.column_stack([year, price, km])           # 7 cars x 3 columns'],
     out:null,
     vis:()=>'<div class="stt">np.column_stack([year, price, km])</div>'+tbl(['','year','price','km'],carRows())+'<div class="eq fi">each array became <b>one column</b><br>each car is now <b>one row</b></div>',
     note:'<code>np.column_stack</code> takes 1-D arrays and places each one as a <b>column</b> of a new 2-D array. Now all the information about one car sits in one <b>row</b>. This table layout (rows = samples, columns = features) is exactly what ML libraries expect.'},

    {pill:'shape',lines:['print(cars.shape)'],
     out:'(7, 3)',
     vis:()=>'<div class="stt">cars.shape</div><div style="color:#38bdf8;font-size:12px">3 columns</div>'+tbl(['','year','price','km'],carRows(),()=> 'grn')+'<div class="row2"><div class="eq fi">rows: <b>7</b> cars<br>columns: <b>3</b> features</div></div><div class="big fi">(7, 3)</div>',
     note:'<code>.shape</code> is a tuple <b>(rows, columns)</b>. It is the quickest check that stacking did what you expected: 7 cars, 3 features.'},

    {pill:'column means',lines:['print("column means (year, price, km):", cars.mean(axis=0).round(1))'],
     out:'column means (year, price, km): '+npArr(cmean.map(v=>rnd(v,1))),
     vis:()=>{
       const r=carRows();
       return '<div class="stt">cars.mean(axis=0)</div>'+tbl(['','year','price','km'],r.concat([
         ['mean',cmean[0].toFixed(2),cmean[1].toFixed(2),cmean[2].toFixed(2)],
         ['.round(1)',rnd(cmean[0],1).toFixed(1),rnd(cmean[1],1).toFixed(1),rnd(cmean[2],1).toFixed(1)]]),(i)=>i>=7?'hl':'')+
         '<div style="color:#facc15;font-size:13px">▼ &nbsp;▼ &nbsp;▼ &nbsp; axis=0 means: go down the rows, one result per column</div>';},
     note:'<code>axis=0</code> averages <b>down the rows</b>, giving one mean per column. Without <code>axis</code> you would get one number mixing years and kilometres, which means nothing. <code>.round(1)</code> keeps one decimal. Notice how different the scales are (2019 vs 65571): this is why features are standardised later.'},

    {pill:'median + std',lines:['','print("median:", np.median(price), "| std:", price.std().round(2))'],
     out:'median: '+pyf(med)+' | std: '+pyf(rnd(sd,2)),
     vis:()=>{
       const mid=3;
       const dev=price.map(p=>({x:p,d:p-mP,s:(p-mP)*(p-mP)}));
       const sumsq=dev.reduce((a,b)=>a+b.s,0);
       return '<div class="stt">median = middle value after sorting</div>'+
         chips('sorted',sp,(i)=>i===mid?'y':'',(i)=>'#'+i)+
         '<div class="eq fi">7 values, so the median is the <b>4th</b>: <span class="y">'+med+'</span><br>mean is '+mP.toFixed(2)+' (pulled up by 95), median is not</div>'+
         '<div class="stt" style="margin-top:10px">std = typical distance from the mean</div>'+
         tbl(['price','x − mean','(x − mean)²'],dev.map(d=>[d.x,d.d.toFixed(2),d.s.toFixed(1)]))+
         eq('sum = '+sumsq.toFixed(1)+' &nbsp;÷ 7 = '+(sumsq/N).toFixed(1)+' (variance)<br>√ '+(sumsq/N).toFixed(1)+' = <b>'+sd.toFixed(4)+'</b> → <b>'+rnd(sd,2)+'</b>');},
     note:'<code>np.median</code> sorts and takes the middle: <b>'+med+'</b>, unaffected by the 95. <code>price.std()</code> averages the squared distances from the mean and takes the square root. numpy divides by <b>n</b> (population std, ddof=0); pandas <code>.std()</code> divides by n−1, so the two differ slightly. The big std ('+rnd(sd,2)+') already hints at the outlier.'},

    {pill:'q1, q3',lines:['q1, q3 = np.percentile(price, [25, 75])'],
     out:null,
     vis:()=>{
       const p1=(N-1)*0.25, p3=(N-1)*0.75;
       return '<div class="stt">np.percentile(price, [25, 75])</div>'+
         chips('sorted',sp,(i)=>(i===1||i===2)?'b':((i===4||i===5)?'g':''),(i)=>'#'+i)+
         '<div class="row2">'+
         eq('<b>25%</b>: position = 0.25 × (7−1) = <span class="y">'+p1+'</span><br>halfway between #1 ('+sp[1]+') and #2 ('+sp[2]+')<br>q1 = <b>'+q1+'</b>')+
         eq('<b>75%</b>: position = 0.75 × (7−1) = <span class="y">'+p3+'</span><br>halfway between #4 ('+sp[4]+') and #5 ('+sp[5]+')<br>q3 = <b>'+q3+'</b>')+'</div>'+
         nl({dots:price,col:(i)=>i===6?'#f87171':'#38bdf8',marks:[{x:q1,t:'q1 '+q1,c:'#38bdf8'},{x:q3,t:'q3 '+q3,c:'#4ade80',row:1}]});},
     note:'A percentile is a value that a given share of the data lies below. numpy sorts the data and, when the position falls <b>between</b> two items, interpolates linearly. <code>q1, q3 = ...</code> <b>unpacks</b> the two returned numbers into two names. Quartiles: 25% of the cars cost less than <b>'+q1+'</b>, 75% less than <b>'+q3+'</b>.'},

    {pill:'iqr',lines:['iqr = q3 - q1'],
     out:null,
     vis:()=>'<div class="stt">iqr = q3 − q1 (the interquartile range)</div>'+nl({dots:price,col:(i)=>i===6?'#f87171':'#38bdf8',band:{a:q1,b:q3,c:'#4ade80'},marks:[{x:q1,t:'q1 '+q1,c:'#38bdf8'},{x:q3,t:'q3 '+q3,c:'#4ade80'}]})+eq(q3+' − '+q1+' = <b>'+iqr+'</b><br>the green band holds the <b>middle half</b> of the cars')+'<div class="eq fi" style="margin-left:6px">the 95 is far outside it</div>',
     note:'The <b>IQR</b> is the width of the middle 50% of the data. Unlike the std it ignores extreme values completely, so one huge price cannot distort it. That is what makes it a good yardstick for deciding what counts as unusual.'},

    {pill:'outlier mask',lines:['outliers = (price < q1 - 1.5 * iqr) | (price > q3 + 1.5 * iqr)'],
     out:null,
     vis:()=>{
       return '<div class="stt">fences: q1 − 1.5·iqr and q3 + 1.5·iqr</div>'+
         eq('lower fence = '+q1+' − 1.5 × '+iqr+' = <b>'+lowF+'</b><br>upper fence = '+q3+' + 1.5 × '+iqr+' = <b>'+upF+'</b>')+
         nl({dots:price,col:(i)=>mask[i]?'#f87171':'#38bdf8',band:{a:Math.max(0,lowF),b:upF,c:'#4ade80'},marks:[{x:lowF,t:'lower '+lowF,c:'#facc15'},{x:upF,t:'upper '+upF,c:'#facc15',row:1}]})+
         tbl(['price','< lower?','> upper?','OR'],price.map((p,i)=>[p,p<lowF?'True':'False',p>upF?'True':'False',mask[i]?'<b>True</b>':'False']),(i)=>mask[i]?'red':'');},
     note:'The <b>1.5 × IQR rule</b>: anything more than 1.5 IQRs below q1 or above q3 is flagged. Both comparisons are done on <b>every price at once</b>, giving two True/False arrays. <code>|</code> is the <b>element-wise OR</b> (the word <code>or</code> would fail on arrays), and the brackets are required because <code>|</code> binds tighter than <code>&lt;</code>. Only car 6 (price 95) is above the upper fence.'},

    {pill:'print mask',lines:['print("IQR outlier mask:", outliers)'],
     out:'IQR outlier mask: '+boolArr(mask),
     vis:()=>'<div class="stt">outliers is a boolean array, one True/False per car</div>'+chips('price',price,(i)=>mask[i]?'r':'')+chips('outliers',mask.map(b=>b?'True':'False'),(i)=>mask[i]?'r':'g')+eq('use it as a filter:<br><span class="y">price[outliers]</span> → ['+price.filter((p,i)=>mask[i]).join(', ')+']<br><span class="y">price[~outliers]</span> → ['+price.filter((p,i)=>!mask[i]).join(', ')+']'),
     note:'The mask has the same length as <code>price</code>. A boolean array like this can be used to <b>select or drop</b> rows, for example <code>cars[~outliers]</code> keeps only the normal cars. Whether to drop or keep an outlier is a decision about your data, not something the code can decide.'},

    {pill:'z-scores',lines:['','z_scores = (price - price.mean()) / price.std()      # standardisation: a core ML preprocessing step'],
     out:null,
     vis:()=>'<div class="stt">z = (price − mean) ÷ std, for every price at once</div>'+eq('mean = '+mP.toFixed(2)+' &nbsp; std = '+sd.toFixed(2))+tbl(['price','price − mean','÷ std = z'],price.map((p,i)=>[p,(p-mP).toFixed(2),z[i].toFixed(2)]),(i)=>mask[i]?'red':''),
     note:'Subtracting a single number from a whole array, then dividing by another, is called <b>broadcasting</b>: numpy applies it to every element, no loop needed. The result says "how many standard deviations from the mean": the new array has mean <b>0</b> and std <b>1</b>. This puts features such as year and km on the same scale, which many ML models need.'},

    {pill:'print z',lines:['print(z_scores.round(2))'],
     out:npArr(z.map(v=>rnd(v,2))),
     vis:()=>{
       const cx=320,u=110;
       let s='<svg viewBox="0 0 640 250"><line x1="'+cx+'" y1="10" x2="'+cx+'" y2="235" stroke="#64748b" stroke-dasharray="4 3"/>';
       for(let t=-2;t<=2;t++) s+='<text x="'+(cx+t*u)+'" y="248" fill="#64748b" font-size="10" text-anchor="middle">'+t+'</text>';
       z.forEach((v,i)=>{
         const y=14+i*31, x=v>=0?cx:cx+v*u, w=Math.abs(v)*u, c=mask[i]?'#f87171':(v>=0?'#4ade80':'#38bdf8');
         s+='<text x="8" y="'+(y+15)+'" fill="#94a3b8" font-size="11">car '+i+' ('+price[i]+')</text><rect x="'+x+'" y="'+y+'" width="'+w+'" height="22" rx="4" fill="'+c+'"/>'+
            '<text x="'+(v>=0?cx+w+6:cx+v*u-6)+'" y="'+(y+16)+'" fill="#e2e8f0" font-size="11" text-anchor="'+(v>=0?'start':'end')+'">'+rnd(v,2)+'</text>';
       });
       return '<div class="stt">z-scores: distance from the mean in std units</div>'+s+'</svg>';},
     note:'Negative z = below the average price, positive = above. Six cars sit within about ±1 of the mean, while the 95 is <b>'+rnd(z[6],2)+'</b> stds above. A caution: with only 7 values, no z-score can ever exceed about 2.45, so the common "|z| &gt; 3" rule could never flag anything here. The IQR rule did, which is why it is more robust for small or skewed data.'},

    {pill:'correlation',lines:['print("correlation of year and price:", np.corrcoef(year, price)[0, 1].round(2))'],
     out:'correlation of year and price: '+pyf(rnd(rr,2)),
     vis:()=>{
       const X=v=>50+(v-2015.5)/7*560, Y=v=>220-(v/100)*200;
       const my=mean(year), b=year.reduce((s,v,i)=>s+(v-my)*(price[i]-mP),0)/year.reduce((s,v)=>s+(v-my)*(v-my),0), a=mP-b*my;
       let s='<svg viewBox="0 0 640 250"><line x1="50" y1="220" x2="610" y2="220" stroke="#475569"/><line x1="50" y1="20" x2="50" y2="220" stroke="#475569"/>';
       for(let t=2016;t<=2022;t++) s+='<text x="'+X(t)+'" y="238" fill="#64748b" font-size="10" text-anchor="middle">'+t+'</text>';
       for(let t=0;t<=100;t+=25) s+='<text x="42" y="'+(Y(t)+3)+'" fill="#64748b" font-size="10" text-anchor="end">'+t+'</text>';
       s+='<line x1="'+X(2016)+'" y1="'+Y(a+b*2016)+'" x2="'+X(2022)+'" y2="'+Y(a+b*2022)+'" stroke="#facc15" stroke-dasharray="6 4" stroke-width="2"/>';
       year.forEach((y,i)=>{s+='<circle cx="'+X(y)+'" cy="'+Y(price[i])+'" r="7" fill="'+(mask[i]?'#f87171':'#38bdf8')+'" stroke="#0f172a" stroke-width="2"/>';});
       s+='<text x="330" y="12" fill="#94a3b8" font-size="11" text-anchor="middle">year (x) vs price (y)</text></svg>';
       return '<div class="stt">np.corrcoef(year, price)</div>'+s+
         '<div class="row2">'+eq('corrcoef returns a <b>2 × 2</b> matrix:<br>[[ 1.00 &nbsp; <span class="y">'+rr.toFixed(2)+'</span> ]<br>&nbsp;[ <span class="y">'+rr.toFixed(2)+'</span> &nbsp; 1.00 ]]')+eq('<b>[0, 1]</b> picks row 0, column 1<br>r = <b>'+rnd(rr,2)+'</b>')+'</div>';},
     note:'<code>np.corrcoef</code> returns a <b>matrix</b> of correlations between every pair: the diagonal is 1 (a variable with itself), and <code>[0, 1]</code> is the entry for year vs price. <b>r</b> runs from −1 (perfect opposite trend) through 0 (no linear link) to +1 (perfect upward trend). Newer cars cost more here, r = <b>'+rnd(rr,2)+'</b>, but the outlier is doing much of that work: correlation is also sensitive to extreme values.'}
  ];

  // ---------------- cumulative rendering ----------------
  const outs=[]; STEPS.forEach(s=>{ if(s.out!==null) outs.push(s.out); });
  let k=0,timer=null;
  function draw(){
    // code
    let n=0,h='';
    STEPS.forEach((s,i)=>{
      if(i>k) return;
      s.lines.forEach(t=>{
        n++;
        const c=t.indexOf('#'), body=c>-1?esc(t.slice(0,c))+'<span class="cm">'+esc(t.slice(c))+'</span>':esc(t);
        h+='<div class="sl '+(i===k?'on':'old')+'"><i>'+n+'</i>'+body+'</div>';
      });
    });
    $('Cxcode').innerHTML=h;
    $('Cxcode').scrollTop=$('Cxcode').scrollHeight;
    // terminal
    let th='';
    STEPS.forEach((s,i)=>{ if(i<=k && s.out!==null) th+='<div class="l'+(i===k?' new':'')+'">'+esc(s.out)+'</div>'; });
    $('Cxterm').innerHTML=th||'<div class="l">(nothing printed yet)</div>';
    // stage + note
    $('Cxstage').innerHTML=STEPS[k].vis();
    $('Cxnote').innerHTML='<b style="color:#38bdf8">Step '+(k+1)+' of '+STEPS.length+'</b><br>'+STEPS[k].note;
    // pills
    $('Cxpills').innerHTML=STEPS.map((s,i)=>'<span class="pill '+(i===k?'on':(i<k?'done':''))+'" data-i="'+i+'">'+(i+1)+' '+esc(s.pill)+'</span>').join('');
    Array.prototype.forEach.call($('Cxpills').children,e=>{e.onclick=()=>{stop();k=Number(e.getAttribute('data-i'));draw();};});
    $('Cxc').textContent=' '+(k+1)+' / '+STEPS.length;
  }
  function stop(){ if(timer){clearInterval(timer);timer=null;} $('Cxp').textContent='⏵ Play'; }
  $('Cxn').onclick=()=>{stop();k=Math.min(STEPS.length-1,k+1);draw();};
  $('Cxb').onclick=()=>{stop();k=Math.max(0,k-1);draw();};
  $('Cxr').onclick=()=>{stop();k=0;draw();};
  $('Cxp').onclick=()=>{
    if(timer){stop();return;}
    if(k>=STEPS.length-1)k=0;
    $('Cxp').textContent='⏸ Pause'; draw();
    timer=setInterval(()=>{ if(k>=STEPS.length-1){stop();return;} k++; draw(); },5500);
  };
  draw();
})();
</script>
'''))

## 25.6 Missing values: `np.nan`
Scraped data always has gaps. Missing numbers are `np.nan`; ordinary statistics return `nan` if one value is missing, so use the `nan` versions.

In [ ]:
km_scraped = np.array([62_000, np.nan, 31_000, np.nan, 12_000])
print("missing:", np.isnan(km_scraped))
print("mean:", km_scraped.mean(), "| nanmean:", np.nanmean(km_scraped))

filled = np.where(np.isnan(km_scraped), np.nanmedian(km_scraped), km_scraped)    # fill gaps with the median
print(filled)

missing: [False  True False  True False]
mean: nan | nanmean: 35000.0
[62000. 31000. 31000. 31000. 12000.]


### Exercise: Value and Running Cost over 5 Years

A buyer compares a **petrol** car (14 km per litre) and a **hybrid** (22 km per litre) that costs Rs. 400,000 more. Assume petrol at Rs. 270 per litre and 15,000 km driven per year (illustrative numbers).
1. With NumPy, compute the **cumulative fuel cost** for years 1 to 5 for both cars.
2. Add the price difference and find in **which year** the hybrid becomes cheaper in total.
3. A car loses 8% of its value every year. Compute the value of a Rs. 3,000,000 car for ages 0 to 10 and print the ages when it is worth less than half.

In [ ]:
# Your code here

In [ ]:
#@title Solution { display-mode: "form" }

import numpy as np
years = np.arange(1, 6)
km_per_year, petrol_price = 15_000, 270

petrol_cost = km_per_year / 14 * petrol_price * years            # cumulative litres x price
hybrid_cost = km_per_year / 22 * petrol_price * years + 400_000
print("Petrol cumulative:", petrol_cost.round(0))
print("Hybrid cumulative:", hybrid_cost.round(0))
cheaper = years[hybrid_cost < petrol_cost]
print("Hybrid cheaper from year:", cheaper[0] if cheaper.size else "never within 5 years")

ages = np.arange(0, 11)
value = 3_000_000 * (1 - 0.08) ** ages
print(value.round(0))
print("Worth less than half at ages:", ages[value < 1_500_000])

---
# MODULE 26: Pandas

**Pandas** adds labels to NumPy: a **Series** is one labelled column, a **DataFrame** is a table. It reads files, filters, groups, joins and cleans data in a few lines. We start with a small hand-made table of listings.

## 26.1 Create and inspect a DataFrame

In [ ]:
import pandas as pd

demo = pd.DataFrame({
    "title":      ["Suzuki Alto VXR 2019", "Toyota Corolla Altis 2018", "Honda Civic Oriel 2020", "Suzuki Cultus VXL 2021",
                   "Toyota Aqua 2017", "Honda City Aspire 2022", "Kia Sportage Alpha 2021", "Toyota Corolla GLi 2016"],
    "city":       ["Lahore", "Karachi", "Islamabad", "Lahore", "Karachi", "Islamabad", "Lahore", "Karachi"],
    "year":       [2019, 2018, 2020, 2021, 2017, 2022, 2021, 2016],
    "km":         [61_000, 82_000, 38_000, 29_000, 70_000, 21_000, 44_000, 110_000],
    "fuel":       ["Petrol", "Petrol", "Petrol", "Petrol", "Hybrid", "Petrol", "Petrol", "Petrol"],
    "price_lacs": [28.5, 58.0, 72.0, 41.0, 52.0, 59.5, 88.0, 39.0],
})

print(demo.shape)
print(demo.dtypes)
demo.head(3)

(8, 6)
title          object
city           object
year            int64
km              int64
fuel           object
price_lacs    float64
dtype: object


,title,city,year,km,fuel,price_lacs
0,Suzuki Alto VXR 2019,Lahore,2019,61000,Petrol,28.5
1,Toyota Corolla Altis 2018,Karachi,2018,82000,Petrol,58.0
2,Honda Civic Oriel 2020,Islamabad,2020,38000,Petrol,72.0


In [ ]:
demo.info()
demo.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   title       8 non-null      object 
 1   city        8 non-null      object 
 2   year        8 non-null      int64  
 3   km          8 non-null      int64  
 4   fuel        8 non-null      object 
 5   price_lacs  8 non-null      float64
dtypes: float64(1), int64(2), object(3)
memory usage: 516.0+ bytes


,year,km,price_lacs
count,8.00000,8.000000,8.000000
mean,2019.25000,56875.000000,54.750000
std,2.12132,29830.174081,19.160972
min,2016.00000,21000.000000,28.500000
25%,2017.75000,35750.000000,40.500000
50%,2019.50000,52500.000000,55.000000
75%,2021.00000,73000.000000,62.625000
max,2022.00000,110000.000000,88.000000


## 26.2 Select and filter
`df["col"]` gives a Series; `df[["a", "b"]]` a DataFrame; `.loc` selects by **label**, `.iloc` by **position**; a boolean mask filters rows.

In [ ]:
print(demo["price_lacs"].mean())
print(demo[["title", "price_lacs"]].head(3))
print(demo.loc[0, "title"], "|", demo.iloc[0, 0])                      # label vs position
print(demo.loc[2:4, ["title", "km"]])                                  # .loc slices include the end label
# print(demo.iloc[2:4, 0:2])
lahore = demo[demo["city"] == "Lahore"]
cheap_recent = demo[(demo["price_lacs"] < 60) & (demo["year"] >= 2018)]
print(len(lahore), len(cheap_recent))
# print(demo.query("fuel == 'Petrol' and km < 50000")["title"].tolist())
# print(demo[demo["city"].isin(["Karachi", "Islamabad"])].shape)
# print(demo.sort_values("price_lacs", ascending=False).head(3)[["title", "price_lacs"]])
# print(demo.nsmallest(2, "km")[["title", "km"]])

54.75
                       title  price_lacs
0       Suzuki Alto VXR 2019        28.5
1  Toyota Corolla Altis 2018        58.0
2     Honda Civic Oriel 2020        72.0
Suzuki Alto VXR 2019 | Suzuki Alto VXR 2019
                    title     km
2  Honda Civic Oriel 2020  38000
3  Suzuki Cultus VXL 2021  29000
4        Toyota Aqua 2017  70000
3 4


## 26.3 New columns, text and `apply`
Operate on whole columns at once. The `.str` accessor applies string methods and **regular expressions** to every row; you will use this heavily when cleaning scraped text.

In [ ]:
demo = demo.assign(age=2026 - demo["year"])
demo["km_per_year"] = (demo["km"] / demo["age"]).round(0)
demo["make"] = demo["title"].str.split().str[0]                        # first word of the title
demo["is_toyota"] = demo["title"].str.contains("Toyota")
demo["title_year"] = demo["title"].str.extract(r"(\d{4})")[0].astype(int)   # regex: four digits

segment = demo["price_lacs"].apply(lambda p: "budget" if p < 40 else "mid" if p < 70 else "premium")
demo["segment"] = segment
print(demo[["title", "age", "km_per_year", "make", "segment"]])

                       title  age  km_per_year    make  segment
0       Suzuki Alto VXR 2019    7       8714.0  Suzuki   budget
1  Toyota Corolla Altis 2018    8      10250.0  Toyota      mid
2     Honda Civic Oriel 2020    6       6333.0   Honda  premium
3     Suzuki Cultus VXL 2021    5       5800.0  Suzuki      mid
4           Toyota Aqua 2017    9       7778.0  Toyota      mid
5     Honda City Aspire 2022    4       5250.0   Honda      mid
6    Kia Sportage Alpha 2021    5       8800.0     Kia  premium
7    Toyota Corolla GLi 2016   10      11000.0  Toyota   budget


## 26.4 `groupby`, `pivot_table`, `value_counts`
Split the data into groups, calculate something per group, combine the results. This is the same idea as SQL `GROUP BY`.

In [ ]:
print(demo.groupby("city")["price_lacs"].median())

summary = demo.groupby("make").agg(
    listings=("title", "count"),
    median_price=("price_lacs", "median"),
    avg_km=("km", "mean"),
).sort_values("median_price", ascending=False)
print(summary)

print(demo.pivot_table(values="price_lacs", index="city", columns="fuel", aggfunc="median"))
print(demo["city"].value_counts())
print(pd.crosstab(demo["city"], demo["segment"]))

## 26.5 Missing values and duplicates

In [ ]:
messy = pd.concat([demo[["title", "km", "price_lacs"]].head(4),
                   pd.DataFrame({"title": ["Suzuki Alto VXR 2019", "Honda BR-V 2020"], "km": [61_000, None], "price_lacs": [28.5, None]})],
                  ignore_index=True)
print(messy.isna().sum())
print("duplicates:", messy.duplicated().sum())

deduped = messy.drop_duplicates()
filled = deduped.assign(km=deduped["km"].fillna(deduped["km"].median()))      # fill: a numeric gap with the median
dropped = filled.dropna(subset=["price_lacs"])                                  # drop: rows without the target value
print(dropped)

## 26.6 Combining tables: `merge` and `concat`
`merge` is SQL's `JOIN`: match rows on a shared column.

In [ ]:
city_info = pd.DataFrame({"city": ["Lahore", "Karachi", "Islamabad"], "province": ["Punjab", "Sindh", "Islamabad Capital Territory"]})
with_province = demo.merge(city_info, on="city", how="left")
print(with_province[["title", "city", "province"]].head(4))
print(with_province.groupby("province")["price_lacs"].median())

## 26.7 Dates and files

In [ ]:
listed = pd.to_datetime(pd.Series(["2026-10-20", "2026-11-01", "2026-11-09"]))
days_online = (pd.Timestamp("2026-11-13") - listed).dt.days
print(days_online.tolist(), listed.dt.month.tolist(), listed.dt.day_name().tolist())

demo.to_csv("demo_cars.csv", index=False)                  # write
print(pd.read_csv("demo_cars.csv").shape)                  # read it back

### Exercise: Dealer Inventory Report

A dealer exports the current stock. Using the `inventory` table below:
1. The **average age** of the stock per fuel type.
2. The **two most expensive** cars in each city (hint: sort, then `groupby(...).head(2)`).
3. Add a column `overpriced` that is `True` when a car costs more than **120% of the median price of its city**.
4. Save the overpriced cars to `overpriced.csv`.

In [ ]:
inventory = pd.DataFrame({
    "title":      ["Alto VXR", "Corolla GLi", "Civic Oriel", "Cultus VXL", "Aqua", "City Aspire", "Sportage Alpha", "Yaris ATIV", "Vitz", "Wagon R"],
    "city":       ["Lahore", "Lahore", "Lahore", "Karachi", "Karachi", "Karachi", "Islamabad", "Islamabad", "Islamabad", "Lahore"],
    "fuel":       ["Petrol", "Petrol", "Petrol", "Petrol", "Hybrid", "Petrol", "Petrol", "Petrol", "Petrol", "Petrol"],
    "age":        [7, 10, 6, 5, 9, 4, 5, 3, 8, 4],
    "price_lacs": [28.5, 39.0, 72.0, 41.0, 52.0, 59.5, 88.0, 50.0, 31.0, 30.5],
})
# Your code here

In [ ]:
#@title Solution { display-mode: "form" }

print(inventory.groupby("fuel")["age"].mean().round(1))

top2 = inventory.sort_values("price_lacs", ascending=False).groupby("city").head(2).sort_values(["city", "price_lacs"], ascending=[True, False])
print(top2)

city_median = inventory.groupby("city")["price_lacs"].transform("median")      # same length as inventory
inventory["overpriced"] = inventory["price_lacs"] > 1.2 * city_median
print(inventory[inventory["overpriced"]])
inventory[inventory["overpriced"]].to_csv("overpriced.csv", index=False)

---
# MODULE 27: Web Scraping

**Web scraping** means collecting data from web pages with code. It is powerful, and it comes with responsibilities, so we start with the rules.

## 27.1 Scrape responsibly
Being able to download a page does not mean you are allowed to collect it. Before writing any scraper:

| Check | Why it matters |
|---|---|
| **Is there an official API or a downloadable dataset?** | It is faster, stable and permitted. Prefer it every time. |
| **Read the site's Terms of Use** | Many sites forbid automated collection or reuse of their content. If they do, do not scrape. Ask for permission instead. |
| **Read `robots.txt`** (`site.com/robots.txt`) | The site lists the paths robots should not visit, and sometimes a `Crawl-delay`. Respect it. |
| **Limit your load** | Pause between requests, collect only what you need, **cache** pages so you never download the same page twice. |
| **Identify yourself** | Send an honest `User-Agent` with a contact address. |
| **Stop when told to** | Status `403` or `429` means "go away" or "slow down". Stop. Never try to get around blocks, logins or CAPTCHAs. |
| **Respect people's data** | Do not collect phone numbers, names or other personal data. Do not republish a site's content. |
| **Keep it small and educational** | Public does not mean free to reuse. Use scraped data for learning and analysis, not for resale. |

> **For this class:** the instructor decides whether a live scrape is appropriate after reading the site's current Terms of Use and `robots.txt`, and runs it **once**. Students then work from the saved files. The notebook is built so everything also works offline, with simulated listings that look like real scraped pages. When you use the simulated data, the market numbers are **not real**: only the method is.

## 27.2 HTML in five minutes
A web page is **HTML**: a tree of nested **tags**. A tag can have **attributes** (`class`, `id`, `href`) and text.

```html
<ul class="listings">                              <!-- ul = list, class is an attribute -->
  <li class="car" data-id="101">                   <!-- li = list item -->
    <a href="/cars/corolla-2018">Toyota Corolla 2018</a>   <!-- a = link, href = destination -->
    <span class="price">PKR 55.5 lacs</span>
  </li>
</ul>
```
Common tags: `div`, `span` (containers), `h1`..`h6` (headings), `a` (link), `ul/ol/li` (lists), `table/tr/th/td` (tables), `p` (paragraph). In your browser, **right-click a page, Inspect** shows the HTML of whatever you click on: that is how you find what to extract.

## 27.3 BeautifulSoup: parse and search

In [ ]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>Mini Car Market</title></head>
<body>
  <h1 id="page-title">Used cars today</h1>
  <ul class="listings">
    <li class="car" data-id="101"><a href="/cars/corolla-2018">Toyota Corolla 2018</a> <span class="price">PKR 55.5 lacs</span></li>
    <li class="car" data-id="102"><a href="/cars/alto-2019">Suzuki Alto 2019</a> <span class="price">PKR 28.5 lacs</span></li>
    <li class="car sold" data-id="103"><a href="/cars/civic-2020">Honda Civic 2020</a> <span class="price">Call for price</span></li>
  </ul>
  <a rel="next" href="?page=2">Next</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

print(soup.title.get_text())                                  # first <title>
print(soup.find("h1")["id"])                                  # an attribute of a tag
cars = soup.find_all("li", class_="car")                       # all matching tags
print(len(cars), "listings")

for li in cars:
    link = li.find("a")
    print(li["data-id"], "|", link.get_text(strip=True), "|", link["href"], "|", li.find("span", class_="price").get_text(strip=True))

In [ ]:
# CSS selectors: the same ideas you see in browser tools
print([a.get_text() for a in soup.select("ul.listings li a")])
print(soup.select_one("li.sold")["data-id"])
print(soup.select_one("a[rel=next]")["href"])

# Real pages are messy: a tag may be missing. find() returns None, so guard before using the result
li = soup.find("li", class_="car")
badge = li.find("span", class_="badge")
print("badge:", badge.get_text() if badge else "not present")

# Build a list of dictionaries: ready for a DataFrame
records = [{"id": li["data-id"], "title": li.a.get_text(strip=True),
            "price_text": li.find("span", class_="price").get_text(strip=True)} for li in cars]
print(pd.DataFrame(records))

## 27.6 Case study: used-car listings
A listing page shows 25 cars. Each **card** contains a title (with a link to the ad), a price, and a short list of facts (city, year, mileage, fuel, engine size, transmission), plus "Updated 3 days ago". Further pages are reached by changing a page number in the URL, commonly `?page=2`, `?page=3`, and so on (**pagination**). Click "Next" on the real site and read the address bar to confirm the pattern.

**A robust parsing idea.** Class names such as `price-details` change whenever a site is redesigned, and they differ from site to site. Instead of trusting names, our parser finds each **ad link**, climbs to the card around it, and **classifies every list item by what the text looks like** (`2018` is a year, `61,000 km` is mileage, `1300 cc` is an engine). Text patterns survive redesigns better than class names.

**Offline sample site.** The next cell builds simulated listing pages in a folder, using the same kind of card layout. Run it without reading it: it only prepares the practice data (about 1,200 listings over 50 pages). **These listings are simulated, not real PakWheels data.**

In [ ]:
import re, html as htmllib
import numpy as np
import pandas as pd

def simulate_car_listings(n=1200, seed=11):
    """Simulate raw (messy) used-car listings as text, like a scraper would see them."""
    rng = np.random.default_rng(seed)
    # make, model, variant, cc, price (lacs) of a 1-year-old car, fuel, share of automatics, popularity
    catalog = [
        ("Suzuki", "Alto", "VXR", 660, 30, "Petrol", .25, 14), ("Suzuki", "Cultus", "VXL", 1000, 36, "Petrol", .30, 10),
        ("Suzuki", "Wagon R", "VXL", 1000, 32, "Petrol", .20, 7), ("Suzuki", "Swift", "DLX", 1300, 45, "Petrol", .45, 5),
        ("Toyota", "Corolla", "GLi", 1300, 55, "Petrol", .35, 11), ("Toyota", "Corolla", "Altis", 1600, 70, "Petrol", .80, 8),
        ("Toyota", "Yaris", "ATIV", 1300, 50, "Petrol", .60, 6), ("Toyota", "Aqua", "S", 1500, 55, "Hybrid", 1.0, 5),
        ("Toyota", "Prius", "S", 1800, 75, "Hybrid", 1.0, 3), ("Toyota", "Prado", "TX", 2700, 200, "Petrol", 1.0, 2),
        ("Honda", "City", "Aspire", 1500, 55, "Petrol", .60, 8), ("Honda", "Civic", "Oriel", 1500, 85, "Petrol", 1.0, 7),
        ("Honda", "BR-V", "S", 1500, 55, "Petrol", .80, 4), ("Honda", "Vezel", "Hybrid", 1500, 65, "Hybrid", 1.0, 4),
        ("Kia", "Sportage", "Alpha", 2000, 85, "Petrol", 1.0, 4), ("Kia", "Picanto", "AT", 1000, 35, "Petrol", .70, 4),
        ("Hyundai", "Tucson", "GLS", 2000, 90, "Petrol", 1.0, 3), ("Hyundai", "Elantra", "GLS", 2000, 60, "Petrol", .90, 2),
        ("Daihatsu", "Mira", "ES", 660, 24, "Petrol", .60, 5), ("Nissan", "Dayz", "Highway Star", 660, 28, "Petrol", 1.0, 3),
        ("Changan", "Alsvin", "Lumiere", 1500, 40, "Petrol", .70, 3), ("MG", "HS", "Essence", 1500, 80, "Petrol", 1.0, 3),
    ]
    weights = np.array([c[7] for c in catalog], float); weights /= weights.sum()
    cities = ["Lahore", "Karachi", "Islamabad", "Rawalpindi", "Faisalabad", "Peshawar", "Multan", "Gujranwala", "Sialkot", "Hyderabad", "Lala musa", "Sargodha"]
    city_p = np.array([25, 22, 14, 8, 6, 5, 4, 4, 3, 3, 3, 3], float); city_p /= city_p.sum()
    city_factor = {"Islamabad": 1.02, "Karachi": 0.99, "Lahore": 1.0}

    rows = []
    for i in range(n):
        make, model, variant, cc, base, fuel, auto_share, _ = catalog[rng.choice(len(catalog), p=weights)]
        age = int(np.clip(rng.gamma(2.2, 3.0), 0, 21)); year = 2026 - age
        expected_km = age * 11_000 + 3_000
        km = max(300, int(expected_km * rng.lognormal(0, 0.35)) // 500 * 500)
        automatic = rng.random() < auto_share
        if model == "Prado" and rng.random() < .35: fuel_i = "Diesel"
        elif make == "Suzuki" and model in ("Alto", "Cultus", "Wagon R") and rng.random() < .12: fuel_i = "CNG"
        else: fuel_i = fuel
        city = str(rng.choice(cities, p=city_p))
        price = base * (0.92 ** max(age - 1, 0)) * (1.10 if age == 0 else 1.0)
        price *= rng.lognormal(0, 0.10) * (1.05 if automatic else 1.0) * np.exp(-0.02 * (km - expected_km) / 10_000)
        price *= city_factor.get(city, 0.97) * (0.97 if fuel_i == "CNG" else 1.0)
        rows.append(dict(make=make, model=model, variant=variant, year=year, km=km, fuel=fuel_i, cc=cc,
                         trans="Automatic" if automatic else "Manual", city=city, price_lacs=max(price, 2.5)))
    df = pd.DataFrame(rows)

    # ---- make the text messy, the way real listings are ----
    def price_text(p, style):
        if style == "call": return "Call for price"
        if p >= 100: return f"PKR {p / 100:.2f} crore"
        if style == "plain": return f"PKR {int(round(p * 100_000, -3)):,}"
        return f"PKR {p:.1f} lacs"
    styles = rng.choice(["lacs", "plain", "call"], n, p=[.83, .15, .02])
    price_lacs = df["price_lacs"].to_numpy().copy()
    slip = rng.random(n) < .008; price_lacs[slip] /= 10                       # decimal slip: 55.0 typed as 5.5
    troll = rng.random(n) < .005                                               # joke prices, set below
    df["price_text"] = [price_text(p, s) for p, s in zip(price_lacs, styles)]
    df.loc[troll, "price_text"] = "PKR 11,111,111"
    df["mileage_text"] = [f"{k:,} km" for k in df["km"]]
    df.loc[rng.random(n) < .02, "mileage_text"] = ""                            # mileage missing
    old = (df["year"] < 2018).to_numpy() & (rng.random(n) < .03); df.loc[old, "mileage_text"] = "1 km"
    df.loc[rng.random(n) < .005, "mileage_text"] = "999,999 km"
    df["year_text"] = df["year"].astype(str)
    df.loc[rng.random(n) < .004, "year_text"] = "2029"                          # a year in the future
    df["engine_text"] = [f"{c} cc" for c in df["cc"]]
    df["transmission"] = df["trans"]; df.loc[rng.random(n) < .02, "transmission"] = ""
    df["city_text"] = df["city"]
    style = rng.choice(["same", "lower", "upper", "space"], n, p=[.92, .03, .02, .03])
    df.loc[style == "lower", "city_text"] = df["city"].str.lower()
    df.loc[style == "upper", "city_text"] = df["city"].str.upper()
    df.loc[style == "space", "city_text"] = df["city"] + " "
    unit = rng.choice(["hour", "day", "month"], n, p=[.30, .55, .15])
    amount = np.where(unit == "hour", rng.integers(1, 23, n), np.where(unit == "day", rng.integers(1, 29, n), rng.integers(1, 4, n)))
    df["updated_text"] = ["Updated " + f"{a} {u}" + ("s" if a > 1 else "") + " ago" for a, u in zip(amount, unit)]
    df["title"] = df["make"] + " " + df["model"] + " " + df["variant"] + " " + df["year"].astype(str) + " for sale in " + df["city"]
    slug = df["title"].str.lower().str.replace(r"[^a-z0-9]+", "-", regex=True)
    df["url"] = "/used-cars/" + slug + "-" + (10_500_000 + np.arange(n) * 7).astype(str)
    dupes = df.sample(int(n * .03), random_state=3)                              # the same ad shown again on another page
    return pd.concat([df, dupes]).sample(frac=1, random_state=5).reset_index(drop=True)


def render_page(rows, page, last_page):
    """Render one listing page as HTML (a simplified layout for practice)."""
    cards = []
    for r in rows:
        facts = [r.city_text, r.year_text, r.mileage_text, r.fuel, r.engine_text, r.transmission]
        items = "".join(f"<li>{htmllib.escape(f)}</li>" for f in facts if f)
        cards.append(f"""
      <li class="classified-listing">
        <div class="car-info">
          <h3><a class="car-name" href="{r.url}">{htmllib.escape(r.title)}</a></h3>
          <div class="price-details">{r.price_text}</div>
          <ul class="search-vehicle-info">{items}</ul>
          <div class="search-bottom">{r.updated_text}</div>
        </div>
      </li>""")
    nxt = f'<a rel="next" href="?page={page + 1}">Next</a>' if page < last_page else ""
    return f"""<!DOCTYPE html><html><head><title>Used Cars for sale - page {page}</title></head><body>
  <div class="promo">Car loans from PKR 0 processing fee. Inspect your car on 200+ checkpoints.</div>
  <ul class="search-results">{''.join(cards)}
  </ul>
  <div class="pagination">{nxt}</div>
</body></html>"""


def build_offline_site(n=1200, per_page=25):
    OFFLINE_DIR.mkdir(exist_ok=True)
    sim = simulate_car_listings(n)
    pages = [sim.iloc[i:i + per_page] for i in range(0, len(sim), per_page)]
    for p, chunk in enumerate(pages, start=1):
        (OFFLINE_DIR / f"page_{p}.html").write_text(render_page(list(chunk.itertuples()), p, len(pages)), encoding="utf-8")
    return len(pages)

print("Offline site built:", build_offline_site(), "pages in", OFFLINE_DIR)

Offline site built: 50 pages in sample_site


In [ ]:
print((OFFLINE_DIR / "page_1.html").read_text(encoding="utf-8")[:1500])

### The parser
Find each ad link, climb to its card, collect the text of the list items, and classify each one by its pattern. Raw text is kept as-is: **scrape first, clean later**.

In [ ]:
AD_LINK = re.compile(r"/used-cars/.+-\d+/?$")
PRICE = re.compile(r"(PKR\s*[\d.,]+\s*(?:lacs?|lakhs?|crores?)?|Call for price)", re.I)
FUELS = {"Petrol", "Diesel", "Hybrid", "CNG", "LPG", "Electric"}
GEARS = {"Automatic", "Manual"}


def find_card(link):
    """Climb from the ad link to the smallest ancestor that holds a price and only this one ad."""
    for parent in link.parents:
        if parent.name in ("body", "html", "[document]"):
            return None
        text = parent.get_text(" ", strip=True)
        ads = {a["href"] for a in parent.find_all("a", href=AD_LINK)}
        if len(ads) > 1:
            return None                                  # we climbed too far: this ancestor contains several ads
        if PRICE.search(text):
            return parent
    return None


def parse_card(link, card, base_url=BASE_URL):
    text = card.get_text(" ", strip=True)
    facts = [li.get_text(strip=True) for li in card.find_all("li")]
    record = {"url": urljoin(base_url, link["href"]), "title": link.get_text(strip=True),
              "price_text": (PRICE.search(text) or [None])[0], "city": None, "year": None, "mileage_text": None,
              "fuel": None, "engine_text": None, "transmission": None,
              "updated_text": (re.search(r"Updated\s+.+?ago", text) or [None])[0]}
    for fact in facts:                                    # classify each fact by what it LOOKS like
        if re.fullmatch(r"(19|20)\d{2}", fact):                record["year"] = fact
        elif re.fullmatch(r"[\d,]+\s*km", fact, re.I):         record["mileage_text"] = fact
        elif re.fullmatch(r"\d{3,4}\s*cc", fact, re.I):        record["engine_text"] = fact
        elif fact in FUELS:                                 record["fuel"] = fact
        elif fact in GEARS:                                 record["transmission"] = fact
        elif re.fullmatch(r"[A-Za-z .'-]+", fact):          record["city"] = fact
    return record


def parse_listings(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    records, seen = [], set()
    for link in soup.find_all("a", href=AD_LINK):
        card = find_card(link)
        if card is None or link["href"] in seen:
            continue
        seen.add(link["href"])
        records.append(parse_card(link, card))
    return records

first_page = parse_listings((OFFLINE_DIR / "page_1.html").read_text(encoding="utf-8"))
print(len(first_page), "listings parsed from page 1")
pd.DataFrame(first_page).head(5)

### Pagination: scrape many pages
Loop over page numbers until a page is empty. Between **real** requests we pause (rate limiting), and we stop at once if the site refuses us. The first cell only checks `robots.txt`; with `LIVE_SCRAPE = False` everything is read from the offline folder.

In [ ]:
def get_page_html(page, live, delay=3.0):
    if live:
        return polite_get(f"{SEARCH_URL}?page={page}", delay)
    path = OFFLINE_DIR / f"page_{page}.html"
    return path.read_text(encoding="utf-8") if path.exists() else None


def scrape_listings(max_pages=50, live=False):
    delay = 3.0
    if live:
        allowed, crawl_delay = robots_allows(SEARCH_URL)
        if not allowed:
            print("robots.txt does not allow this (or could not be read). Not scraping live: using the offline sample site.")
            live = False
        else:
            delay = max(delay, crawl_delay or 0)
    rows = []
    for page in range(1, max_pages + 1):
        try:
            page_html = get_page_html(page, live, delay)
        except ScrapeBlocked as stop:
            print(stop)
            break
        found = parse_listings(page_html) if page_html else []
        if not found:
            print(f"page {page}: no listings, stopping")
            break
        for r in found:
            r["page"], r["source"] = page, ("live" if live else "offline-sample")
        rows.extend(found)
        if live or page <= 3:
            print(f"page {page}: {len(found)} listings")
    print(f"Collected {len(rows)} listings from {rows[-1]['page'] if rows else 0} pages (source: {'live' if live else 'offline sample'})")
    return pd.DataFrame(rows)


MAX_PAGES = 10 if LIVE_SCRAPE else 50             # live scraping: keep it small
cars_raw = scrape_listings(max_pages=MAX_PAGES, live=LIVE_SCRAPE)
cars_raw.to_csv("cars_raw.csv", index=False)       # save: later modules start from this file
cars_raw.head()

**Sharing instead of re-scraping:** if a live scrape was done, the instructor shares `cars_raw.csv` with the class, so the site is contacted once, not once per student. To use a shared file, place it in the Colab files panel and run `cars_raw = pd.read_csv("cars_raw.csv")`.

Look at what we scraped: prices are text (`PKR 28.5 lacs`, `PKR 1.25 crore`, `Call for price`), mileage has commas and `km`, and the update time is a sentence. Turning this into numbers is the job of Module 28.

In [ ]:
print(cars_raw.shape)
print(cars_raw[["price_text", "mileage_text", "engine_text", "updated_text"]].sample(6, random_state=2))
print(cars_raw["price_text"].str.extract(r"(lacs?|crore|Call)", expand=False).value_counts(dropna=False))

### Exercise: Follow the Next Link

Our loop used fixed page numbers. Real scrapers often **follow the "Next" link** until it disappears. Write `scrape_by_next_link(live=False)` that:
1. starts at page 1, parses the listings,
2. looks for `a[rel=next]` in the page and reads its `href` (for example `?page=2`),
3. continues until there is no next link,
4. returns a dictionary `{page_number: number_of_listings}`.
(Use `get_page_html(page, live=False)` to read pages and a regular expression to read the page number from the link.)

In [ ]:
# Your code here

In [ ]:
#@title Solution { display-mode: "form" }

def scrape_by_next_link(live=False, max_pages=100):
    counts, page = {}, 1
    while page <= max_pages:
        page_html = get_page_html(page, live)
        if not page_html:
            break
        counts[page] = len(parse_listings(page_html))
        nxt = BeautifulSoup(page_html, "html.parser").select_one("a[rel=next]")
        if nxt is None:
            break                                          # last page: no Next link
        page = int(re.search(r"page=(\d+)", nxt["href"]).group(1))
    return counts

per_page = scrape_by_next_link()
print(len(per_page), "pages;", sum(per_page.values()), "listings; first pages:", dict(list(per_page.items())[:3]))

---
# MODULE 28: Data Cleaning

Scraped data is **text**. Before any analysis we turn it into correct numbers and categories, and we write down every decision. The workflow:

1. **Audit**: what is wrong?
2. **Parse** text into numbers (price, mileage, engine, year, update time)
3. **Standardise** categories (make, model, city)
4. **Remove** duplicates and **repair or flag** impossible values
5. **Handle** missing values (drop or fill)
6. **Engineer** useful features
7. **Verify** with assertions and save the clean file

## 28.1 Audit the raw data

In [ ]:
import numpy as np
import pandas as pd
import re

cars_raw = pd.read_csv("cars_raw.csv")                 # the file saved at the end of Module 27
print(cars_raw.shape)
cars_raw.info()

In [ ]:
print("Missing values:\n", cars_raw.isna().sum()[lambda s: s > 0], "\n")
print("Duplicate ads (same url):", cars_raw["url"].duplicated().sum())
print("\nCity spellings:", sorted(cars_raw["city"].dropna().unique())[:12], "...")
print("Years outside 1990-2026:", (~cars_raw["year"].between(1990, 2026)).sum())
print("Price formats:", cars_raw["price_text"].str.extract(r"(lacs?|crore|Call)", expand=False).fillna("plain number").value_counts().to_dict())
print(cars_raw["mileage_text"].value_counts().head(3).to_dict())

**Problem log (to fix below):** duplicate ads, prices as text in three formats plus `Call for price`, mileage as text with missing values and placeholders such as `1 km` and `999,999 km`, a future year, inconsistent city spelling and case, missing transmission, update time as a sentence, make and model hidden inside the title.

## 28.2 Parse text into numbers
Write one small function per messy field. Test each on examples before applying it to 1,200 rows.

In [ ]:
def parse_price(text):
    """'PKR 28.5 lacs' -> 2850000.0 | 'PKR 1.25 crore' -> 12500000.0 | 'PKR 2,850,000' -> 2850000.0 | 'Call for price' -> NaN"""
    if not isinstance(text, str) or "call" in text.lower():
        return np.nan
    match = re.search(r"([\d.]+)\s*(lacs?|lakhs?|crores?)?", text.lower().replace(",", ""))
    if not match:
        return np.nan
    value, unit = float(match.group(1)), match.group(2) or ""
    if unit.startswith(("lac", "lakh")):
        value *= 100_000
    elif unit.startswith("crore"):
        value *= 10_000_000
    return value

for example in ["PKR 28.5 lacs", "PKR 1.25 crore", "PKR 2,850,000", "Call for price"]:
    print(f"{example:<18} -> {parse_price(example)}")

In [ ]:
raw = cars_raw.copy()
raw["price_pkr"] = raw["price_text"].apply(parse_price)
raw["mileage_km"] = pd.to_numeric(raw["mileage_text"].str.replace(r"[^\d]", "", regex=True), errors="coerce")   # keep digits only
raw["engine_cc"] = pd.to_numeric(raw["engine_text"].str.extract(r"(\d+)", expand=False))

# 'Updated 3 days ago' -> 3.0 days
UNIT_DAYS = {"hour": 1 / 24, "day": 1, "week": 7, "month": 30, "year": 365}
parts = raw["updated_text"].str.extract(r"(\d+)\s*(hour|day|week|month|year)")
raw["days_since_update"] = pd.to_numeric(parts[0]) * parts[1].map(UNIT_DAYS)

print(raw[["price_text", "price_pkr", "mileage_text", "mileage_km", "updated_text", "days_since_update"]].sample(5, random_state=4))
print(raw[["price_pkr", "mileage_km", "engine_cc", "days_since_update"]].describe().round(0))

## 28.3 Standardise categories
Split the title into make and model, and make city spellings consistent.

In [ ]:
KNOWN_MAKES = ["Mercedes Benz", "Land Rover", "Range Rover", "Suzuki", "Toyota", "Honda", "Kia", "Hyundai", "Daihatsu",
               "Nissan", "Changan", "MG", "BMW", "Audi", "Mitsubishi", "Mazda", "Proton", "FAW", "DFSK"]

def split_title(title):
    """'Toyota Corolla GLi 2018 for sale in Lahore' -> ('Toyota', 'Corolla')"""
    for make in sorted(KNOWN_MAKES, key=len, reverse=True):          # longest first: 'Land Rover' before 'Land'
        if isinstance(title, str) and title.lower().startswith(make.lower() + " "):
            tokens = title[len(make) + 1:].split(" for sale")[0].split()
            if not tokens:
                return make, None
            model = "Wagon R" if tokens[:2] == ["Wagon", "R"] else tokens[0]
            return make, model
    return None, None

raw[["make", "model"]] = raw["title"].apply(lambda t: pd.Series(split_title(t)))
raw["city"] = raw["city"].str.strip().str.title()                   # ' lahore', 'LAHORE' -> 'Lahore'
print(raw[["title", "make", "model", "city"]].head(4))
print("Unparsed titles:", raw["make"].isna().sum(), "| cities now:", raw["city"].nunique())

## 28.4 Duplicates and impossible values
**Decision rules.** An **error** (impossible year, placeholder mileage) is repaired from other columns or treated as missing. A **plausible but extreme** value is kept. A price far from its peers is **suspicious**: we compare each car to the median of cars with the same make, model and age band.

In [ ]:
before = len(raw)
raw = raw.drop_duplicates(subset="url")
print("Duplicate ads removed:", before - len(raw))

# Year: repair impossible values from the title (a second source of the same fact)
title_year = pd.to_numeric(raw["title"].str.extract(r"(\d{4}) for sale", expand=False))
bad_year = ~raw["year"].between(1990, 2026)
print("Impossible years repaired from title:", bad_year.sum())
raw["year"] = raw["year"].where(~bad_year, title_year)
raw["age"] = 2026 - raw["year"]

# Mileage: placeholders are not real readings, treat them as missing
placeholder = ((raw["mileage_km"] <= 1) & (raw["age"] > 3)) | (raw["mileage_km"] >= 500_000)
print("Placeholder mileages set to missing:", placeholder.sum())
raw.loc[placeholder, "mileage_km"] = np.nan

# Price: flag values far from the median of similar cars
raw["age_band"] = pd.cut(raw["age"], bins=[-1, 2, 5, 9, 30], labels=["0-2", "3-5", "6-9", "10+"])
peer_median = raw.groupby(["make", "model", "age_band"], observed=True)["price_pkr"].transform("median")
ratio = raw["price_pkr"] / peer_median
suspicious = (ratio < 0.4) | (ratio > 2.5)
print("Suspicious prices (below 0.4x or above 2.5x of similar cars):", suspicious.sum())
print(raw.loc[suspicious, ["title", "price_text", "price_pkr"]].head(5).to_string(index=False))
raw.loc[suspicious, "price_pkr"] = np.nan                # treated as missing: they are errors, not real prices

## 28.5 Missing values: drop or fill?
| Column | Decision | Reason |
|---|---|---|
| `price_pkr` (what we want to explain) | **drop the row** | filling the target would invent the answer |
| `mileage_km` | **fill** with age x typical km per year, and add a flag column | a car's mileage is roughly predictable from its age |
| `transmission` | **fill** with the most common value for that make and model | usually determined by the model |

In [ ]:
print("Rows before dropping missing prices:", len(raw))
raw = raw.dropna(subset=["price_pkr"]).copy()
print("Rows after:", len(raw))

typical_km_per_year = (raw["mileage_km"] / raw["age"].clip(lower=1)).median()
raw["km_estimated"] = raw["mileage_km"].isna().astype(int)
raw["mileage_km"] = raw["mileage_km"].fillna((raw["age"].clip(lower=1) * typical_km_per_year).round(-3))
print(f"Typical km per year: {typical_km_per_year:,.0f} | mileage estimated for {raw['km_estimated'].sum()} cars")

mode_by_model = (raw.dropna(subset=["transmission"]).groupby(["make", "model"])["transmission"]
                 .agg(lambda s: s.mode().iloc[0]).rename("model_transmission").reset_index())
raw = raw.merge(mode_by_model, on=["make", "model"], how="left")
raw["transmission"] = raw["transmission"].fillna(raw["model_transmission"]).fillna("Manual")
print(raw["transmission"].value_counts().to_dict())

## 28.6 Feature engineering and the final cleaning function
New columns that carry more meaning: `price_lacs` (easier to read), `km_per_year`, `age_band`, `price_segment`. We then put **every step in one function**, so the pipeline is repeatable on next month's scrape, and **assert** that the result is valid.

In [ ]:
def clean_cars(raw_df):
    """Raw scraped listings -> clean, typed DataFrame. Returns (clean_df, cleaning_log)."""
    d, log = raw_df.copy(), []
    def note(step, removed): log.append({"step": step, "rows_affected": int(removed), "rows_left": len(d)})

    n = len(d); d = d.drop_duplicates(subset="url"); note("remove duplicate ads", n - len(d))

    d["price_pkr"] = d["price_text"].apply(parse_price)
    d["mileage_km"] = pd.to_numeric(d["mileage_text"].str.replace(r"[^\d]", "", regex=True), errors="coerce")
    d["engine_cc"] = pd.to_numeric(d["engine_text"].str.extract(r"(\d+)", expand=False))
    parts = d["updated_text"].str.extract(r"(\d+)\s*(hour|day|week|month|year)")
    d["days_since_update"] = pd.to_numeric(parts[0]) * parts[1].map(UNIT_DAYS)
    d[["make", "model"]] = d["title"].apply(lambda t: pd.Series(split_title(t)))
    d["city"] = d["city"].str.strip().str.title()

    title_year = pd.to_numeric(d["title"].str.extract(r"(\d{4}) for sale", expand=False))
    bad_year = ~d["year"].between(1990, 2026); d["year"] = d["year"].where(~bad_year, title_year); note("repair impossible years", bad_year.sum())
    d["age"] = 2026 - d["year"]
    placeholder = ((d["mileage_km"] <= 1) & (d["age"] > 3)) | (d["mileage_km"] >= 500_000)
    d.loc[placeholder, "mileage_km"] = np.nan; note("placeholder mileage -> missing", placeholder.sum())

    d["age_band"] = pd.cut(d["age"], bins=[-1, 2, 5, 9, 30], labels=["0-2", "3-5", "6-9", "10+"])
    peers = d.groupby(["make", "model", "age_band"], observed=True)["price_pkr"].transform("median")
    ratio = d["price_pkr"] / peers
    odd = (ratio < 0.4) | (ratio > 2.5); d.loc[odd, "price_pkr"] = np.nan; note("suspicious prices -> missing", odd.sum())

    n = len(d); d = d.dropna(subset=["price_pkr", "make", "model", "year"]).copy(); note("drop rows without price, make, model or year", n - len(d))

    km_year = (d["mileage_km"] / d["age"].clip(lower=1)).median()
    d["km_estimated"] = d["mileage_km"].isna().astype(int)
    d["mileage_km"] = d["mileage_km"].fillna((d["age"].clip(lower=1) * km_year).round(-3)); note("fill mileage (estimate + flag)", d["km_estimated"].sum())
    modes = (d.dropna(subset=["transmission"]).groupby(["make", "model"])["transmission"].agg(lambda s: s.mode().iloc[0]).rename("tm").reset_index())
    d = d.merge(modes, on=["make", "model"], how="left")
    missing_tm = d["transmission"].isna().sum(); d["transmission"] = d["transmission"].fillna(d["tm"]).fillna("Manual"); note("fill transmission (mode per model)", missing_tm)

    d["price_lacs"] = (d["price_pkr"] / 100_000).round(2)
    d["km_per_year"] = (d["mileage_km"] / d["age"].clip(lower=1)).round(0)
    d["ad_id"] = d["url"].str.extract(r"-(\d+)$", expand=False).astype("int64")
    for col in ("year", "age", "engine_cc"):
        d[col] = d[col].astype(int)

    keep = ["ad_id", "title", "make", "model", "year", "age", "age_band", "mileage_km", "km_estimated", "km_per_year", "engine_cc",
            "fuel", "transmission", "city", "price_pkr", "price_lacs", "days_since_update", "url", "source"]
    return d[keep].reset_index(drop=True), pd.DataFrame(log)


cars_clean, cleaning_log = clean_cars(cars_raw)

# The contract of a clean dataset
assert cars_clean["ad_id"].is_unique, "duplicate ads remain"
assert cars_clean[["price_pkr", "year", "mileage_km", "make", "model", "city", "transmission"]].notna().all().all(), "missing values remain"
assert cars_clean["year"].between(1990, 2026).all(), "impossible years remain"
assert (cars_clean["price_pkr"] > 0).all() and (cars_clean["mileage_km"] > 0).all()
print("All checks passed.", cars_clean.shape)

print(cleaning_log.to_string(index=False))
cars_clean.to_csv("cars_clean.csv", index=False)
cars_clean.head()

### Exercise: Add Two Cleaning Rules

A colleague raises two concerns about `cars_clean`:
1. **Stale ads**: listings last updated more than **60 days ago** may already be sold, so their prices may be out of date.
2. **Heavy use**: cars driven more than **40,000 km per year** (`km_per_year`) are unusual and may be typing errors or taxis.
Count how many cars match each rule and look at five examples of each. Then decide for each rule: **drop** or **keep and flag**? Create `cars_checked` that applies your decisions, with a column `heavy_use` (1 or 0). Print the shapes before and after and explain your decision in a comment.

In [ ]:
# Your code here

In [ ]:
#@title Solution { display-mode: "form" }

stale = cars_clean["days_since_update"] > 60
heavy = cars_clean["km_per_year"] > 40_000
print("stale ads:", stale.sum(), "| heavy use:", heavy.sum())
print(cars_clean.loc[stale, ["title", "price_lacs", "days_since_update"]].head())
print(cars_clean.loc[heavy, ["title", "year", "mileage_km", "km_per_year"]].head())

# Decision: stale ads may show prices from months ago, so for a PRICE analysis we drop them.
# Heavy use can be real (taxis, ride sharing) and is informative, so we KEEP those cars and flag them.
cars_checked = cars_clean.loc[~stale].copy()
cars_checked["heavy_use"] = (cars_checked["km_per_year"] > 40_000).astype(int)
print(cars_clean.shape, "->", cars_checked.shape, "| flagged heavy use:", cars_checked["heavy_use"].sum())

---
# MODULE 29: Exploratory Data Analysis and Visualisation

**EDA** means looking at the data from many angles to understand it **before** modelling: what is typical, what is unusual, and what is related to what. Charts are the main tool, but every chart should answer a question.

| Question | Chart |
|---|---|
| How is one number spread out? | histogram, boxplot |
| How common is each category? | bar chart |
| How are two numbers related? | scatter plot, correlation heatmap |
| How does a number differ between groups? | boxplot or bar by group |
| How does a number change along an ordered axis (age, time)? | line chart |

Remember: our simulated data is **not** the real market. The method is what you take away. With a real scrape, the same code answers the real questions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100

cars = pd.read_csv("cars_clean.csv")
print(cars.shape, "| source:", cars["source"].iloc[0])
cars[["make", "model", "year", "mileage_km", "fuel", "transmission", "city", "price_lacs"]].head()

## 29.1 Overview

In [ ]:
print(cars[["price_lacs", "age", "mileage_km", "engine_cc", "km_per_year", "days_since_update"]].describe().round(1))
print()
print(cars.describe(exclude="number").loc[["count", "unique", "top"]].T.head(8))

## 29.2 One numeric variable at a time
Look at the **shape**: is it symmetric or skewed? Prices are usually **right-skewed** (many cheap cars, a few very expensive ones), so a **log scale** often reveals the structure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(cars["price_lacs"], bins=40, ax=axes[0, 0]).set(title="Price (lacs): right-skewed", xlabel="lacs")
sns.histplot(np.log10(cars["price_lacs"]), bins=40, ax=axes[0, 1], color="C1").set(title="log10 of price: more symmetric", xlabel="log10(lacs)")
sns.histplot(cars["age"], bins=range(0, 23), ax=axes[1, 0], color="C2").set(title="Age of cars (years)")
sns.histplot(cars["mileage_km"], bins=40, ax=axes[1, 1], color="C3").set(title="Mileage (km)")
plt.tight_layout()
plt.show()

print("skewness: price", round(cars["price_lacs"].skew(), 2), "| log price", round(np.log10(cars["price_lacs"]).skew(), 2))
print("mean vs median price:", round(cars["price_lacs"].mean(), 1), "vs", cars["price_lacs"].median())

In [ ]:
q1, q3 = cars["price_lacs"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"IQR upper fence: {upper:.1f} lacs | cars above it: {(cars['price_lacs'] > upper).sum()} (expensive cars: real, not errors)")

plt.figure(figsize=(9, 2.8))
sns.boxplot(x=cars["price_lacs"])
plt.title("Price boxplot: the dots are expensive but genuine cars")
plt.show()

## 29.3 Categories: what does the market look like?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (col, title) in zip(axes.flat, [("make", "Listings by make"), ("city", "Listings by city"), ("fuel", "Fuel type"), ("transmission", "Transmission")]):
    counts = cars[col].value_counts().head(8)
    sns.barplot(x=counts.values, y=counts.index, ax=ax, color="C0")
    ax.set(title=title, xlabel="listings", ylabel="")
plt.tight_layout()
plt.show()

print((cars["transmission"].value_counts(normalize=True) * 100).round(1).to_dict())

## 29.4 Two variables: what is related to price?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.scatterplot(data=cars, x="age", y="price_lacs", alpha=0.35, ax=axes[0]).set(yscale="log", title="Price falls with age (log scale)")
sns.scatterplot(data=cars, x="mileage_km", y="price_lacs", alpha=0.35, color="C1", ax=axes[1]).set(yscale="log", title="Price versus mileage", xlabel="km")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
sns.boxplot(data=cars, x="transmission", y="price_lacs", ax=axes[0]).set(yscale="log", title="Price by transmission")
order = cars.groupby("fuel")["price_lacs"].median().sort_values().index
sns.boxplot(data=cars, x="fuel", y="price_lacs", order=order, ax=axes[1]).set(yscale="log", title="Price by fuel type")
plt.tight_layout()
plt.show()
print("Careful: automatics are also newer and bigger cars on average, so this is not a pure effect. See 29.6.")

## 29.5 Depreciation: how fast do cars lose value?
A line chart along an ordered axis (age). The **median** is used because it is not distorted by extreme prices.

In [ ]:
by_age = cars.groupby("age")["price_lacs"].median()
yearly_change = by_age.pct_change().loc[2:10]
print(f"Typical yearly price change between ages 1 and 10: {yearly_change.median():.1%}")

top_makes = cars["make"].value_counts().head(4).index
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
by_age.plot(marker="o", ax=axes[0]); axes[0].set(title="Median price by age (all cars)", ylabel="lacs", yscale="log")
sns.lineplot(data=cars[cars["make"].isin(top_makes) & (cars["age"] <= 15)], x="age", y="price_lacs", hue="make",
             estimator="median", errorbar=None, marker="o", ax=axes[1])
axes[1].set(title="Median price by age: top makes", yscale="log", ylabel="lacs")
plt.tight_layout()
plt.show()

## 29.6 Answer the owner's questions with `groupby`
Comparing raw averages can mislead: a city may simply have more expensive models, and a make may sell old cheap models and new expensive ones. A fairer comparison measures each car against the **median price of the same make, model and year** (a relative price: 1.00 means typical).

In [ ]:
cars["relative_price"] = cars["price_lacs"] / cars.groupby(["make", "model", "year"])["price_lacs"].transform("median")

# 1. Which models hold their value? Median price of 3-5 year old cars compared with 0-2 year old cars
stats = cars.groupby(["make", "model", "age_band"], observed=True)["price_lacs"].agg(["median", "count"]).unstack()
enough = (stats["count"]["0-2"] >= 8) & (stats["count"]["3-5"] >= 8)             # compare only where both bands have enough ads
retention = (stats["median"]["3-5"] / stats["median"]["0-2"])[enough].sort_values(ascending=False)
retention.index = [f"{make} {model}" for make, model in retention.index]
print("Value retained after about 4 years (higher = holds value better):")
print(retention.round(2))

# 2. City premium: average relative price (cities with enough listings)
big_cities = cars["city"].value_counts()[lambda s: s >= 40].index
city_premium = cars[cars["city"].isin(big_cities)].groupby("city")["relative_price"].mean().sort_values(ascending=False)
print("\nRelative price by city (1.00 = typical for the same make, model and year):")
print(city_premium.round(3))

# 3. Automatic and hybrid premium, like for like
print("\nRelative price by transmission:", cars.groupby("transmission")["relative_price"].mean().round(3).to_dict())
print("Relative price by fuel:", cars.groupby("fuel")["relative_price"].mean().round(3).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
retention.plot.barh(ax=axes[0], color="C2"); axes[0].set(title="Models: value retained after ~4 years", xlabel="median price ratio (3-5y / 0-2y)")
axes[0].invert_yaxis()
city_premium.plot.bar(ax=axes[1], color="C4"); axes[1].axhline(1, color="grey", linestyle="--"); axes[1].set(title="City price premium (like for like)", ylabel="relative price", xlabel="")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 29.7 Correlation: which numbers move together?
Correlation measures **linear** relationships between numbers (-1 to +1). It is not causation, and it ignores categories.

In [ ]:
numeric = cars[["price_lacs", "age", "mileage_km", "engine_cc", "km_per_year", "days_since_update"]]
corr = numeric.corr()

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()

print("Strongest relationship with price:", corr["price_lacs"].drop("price_lacs").abs().idxmax(), "->", corr["price_lacs"].drop("price_lacs").loc[lambda s: s.abs().idxmax()].round(2))
print("age and mileage correlation:", corr.loc["age", "mileage_km"].round(2), "(they carry similar information)")

## 29.8 One-page dashboard and auto-generated insights
Combine the best charts, and let the data write its own headline numbers, so the text is always true for the data you actually have.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sns.histplot(np.log10(cars["price_lacs"]), bins=35, ax=axes[0, 0]).set(title="Price distribution", xlabel="log10(price in lacs)")
sns.scatterplot(data=cars, x="age", y="price_lacs", hue="transmission", alpha=0.4, ax=axes[0, 1]).set(yscale="log", title="Price versus age")
by_age.plot(marker="o", ax=axes[1, 0], color="C3"); axes[1, 0].set(title="Median price by age", yscale="log", ylabel="lacs")
retention.head(8).plot.barh(ax=axes[1, 1], color="C2"); axes[1, 1].invert_yaxis(); axes[1, 1].set(title="Which models hold value", xlabel="3-5y / 0-2y price ratio")
fig.suptitle(f"Used-car market dashboard ({len(cars):,} listings)", fontsize=15)
plt.tight_layout()
fig.savefig("market_dashboard.png", dpi=150)
plt.show()

def auto_insights(df):
    prem = lambda col, a, b: df.groupby(col)["relative_price"].mean().pipe(lambda s: (s[a] / s[b] - 1) if {a, b} <= set(s.index) else np.nan)
    lines = [
        f"Typical (median) price: {df['price_lacs'].median():.1f} lacs; the middle 50% of cars cost {df['price_lacs'].quantile(.25):.1f} to {df['price_lacs'].quantile(.75):.1f} lacs.",
        f"Cars lose about {-yearly_change.median():.0%} of their median price per year between ages 1 and 10.",
        f"Most listed make: {df['make'].value_counts().index[0]} ({df['make'].value_counts(normalize=True).iloc[0]:.0%} of ads); most listings come from {df['city'].value_counts().index[0]}.",
        f"Automatic cars sell at {prem('transmission', 'Automatic', 'Manual'):+.1%} compared with manual cars of the same make, model and year.",
        f"Hybrid cars sell at {prem('fuel', 'Hybrid', 'Petrol'):+.1%} compared with petrol cars of the same make, model and year.",
        f"Price and age correlation: {df['price_lacs'].corr(df['age']):.2f}.",
    ]
    return lines

for line in auto_insights(cars):
    print("-", line)

### Exercise: Shopping List for a Buyer

A customer has a budget of **Rs. 40 lacs** and wants a car that is **at most 6 years old**, with **under 90,000 km** and an **automatic** gearbox.
1. Filter the matching cars.
2. For each make and model, show the number of listings and the median price, keep models with at least 5 listings, and rank them by number of listings.
3. Draw a horizontal bar chart of the **top 8** models by number of listings and write one sentence (in a comment) advising the customer.
4. Bonus: which of the matching cars is the best deal, meaning the **lowest `relative_price`**?

In [ ]:
# Your code here

In [ ]:
#@title Solution { display-mode: "form" }

match = cars[(cars["price_lacs"] <= 40) & (cars["age"] <= 6) & (cars["mileage_km"] < 90_000) & (cars["transmission"] == "Automatic")]
print(len(match), "cars match")

shortlist = (match.groupby(["make", "model"])
                  .agg(listings=("ad_id", "count"), median_price=("price_lacs", "median"), median_km=("mileage_km", "median"))
                  .query("listings >= 5").sort_values("listings", ascending=False))
print(shortlist.head(8))

top8 = shortlist.head(8).reset_index()
plt.figure(figsize=(8, 4))
sns.barplot(data=top8, x="listings", y=top8["make"] + " " + top8["model"], color="C0")
plt.title("Best-supplied cars within the budget"); plt.ylabel("")
plt.show()
# Advice: the most listed models give the most choice and the easiest resale, so start comparing those.

best = match.sort_values("relative_price").head(3)[["title", "price_lacs", "mileage_km", "relative_price"]]
print(best)

---
# Day 4 Summary

| Module | What you can now do |
|---|---|
| NumPy | create arrays, vectorise calculations, use boolean masks, axes, broadcasting and `nan` |
| Pandas | select, filter, group, pivot, merge, handle missing values and dates, read and write files |
| Web scraping | read HTML, use BeautifulSoup and `read_html`, paginate, and scrape **responsibly** (terms, `robots.txt`, rate limits, caching) |
| Data cleaning | parse text into numbers, standardise categories, treat duplicates, errors and gaps, log every decision, verify with assertions |
| EDA and visualisation | describe distributions, find relationships, compare groups fairly, and communicate with charts |

### The data workflow you used
Collect (respectfully) -> **raw file** -> clean with a documented function -> **clean file** -> explore -> insights. Always keep the raw file untouched, so you can rerun the cleaning when rules change.

### Take-home project
Pick one used-vehicle or housing dataset (a public dataset such as one from Kaggle, or a site whose terms allow collection). Build the full pipeline:
1. A scraper or loader with caching and a `Crawl-delay` respect (or a downloaded CSV if scraping is not allowed).
2. A `clean_data()` function with assertions and a cleaning log.
3. A notebook with at least five charts, each with one sentence of insight.
4. Push everything to GitHub with a README (Day 3) that states the data source and its terms of use.

### Next in the ML track
You now have clean, explored data: the starting point of every model. Next: scikit-learn, where a model learns to predict the price of a used car from its age, mileage and make.